# 01 — Exploratory Data Analysis

**Prediction unit:** one client. **Target:** whether that client is associated with electricity/gas consumption fraud.

This notebook establishes data quality, class balance, table relationships, observation windows, and the first client-level behavioral features. The invoice table is processed in chunks so the full 4.5M-row training file does not need to live in memory.

**How to use it:** run the notebook top to bottom once, then discuss the findings marked in the final section before moving to the baseline model. The raw source is read from Parquet for efficient, repeatable local execution; the optional output remains a small CSV with one row per client.

## Stakeholder question and success criteria

Assumption: a utility investigation team has limited capacity and wants a ranked list of clients for inspection. The model should surface fraud while avoiding too many unproductive inspections.

- Primary development metric: **average precision (PR-AUC)**.
- Operational metrics: **precision@k** and **recall@k**, where `k` is inspection capacity.
- Secondary diagnostics: ROC-AUC, confusion matrix, and segment-level performance.

Before final modeling, confirm the cost of an inspection, the cost of missed fraud, inspection capacity, and whether labels refer to a fixed point in time. Those answers determine `k` and the threshold; a probability of 0.5 is not automatically a sensible operational cutoff.

For this first pass, use the charts to form hypotheses—not to declare causes. A difference between fraud and non-fraud clients can come from genuine behavior, data collection, or a longer observation period.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path("data")
PARQUET_DIR = DATA_DIR / "parquet"
CHUNK_SIZE = 250_000
SAVE_FEATURES = False
AS_OF_DATE = None  # Example: "2018-12-31"; required for a real deployment backtest.
FINAL_HOLDOUT_SIZE = 0.20
RANDOM_STATE = 42

required_files = [PARQUET_DIR / "client_train.parquet", PARQUET_DIR / "invoice_train.parquet"]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}. See README.md for data setup.")

print(f"Using Parquet data from: {PARQUET_DIR.resolve()}")

## 1. Understand the two-table data model

`client_train.parquet` has one row per client and contains the target. `invoice_train.parquet` has many dated invoices per client. The safe modeling table therefore has one row per client after invoice aggregation. Splitting invoice rows would put the same client into both train and validation and produce leakage.

The relationship is **one client to many invoices**. Client fields describe who the customer is; invoice fields describe a sequence of readings and consumption events. In this notebook we turn that sequence into summary features such as history length, mean consumption, zero-consumption rate, activity span, and electricity/gas mix. Later iterations can add trends and recent-versus-historical changes.

In [ ]:
client = pd.read_parquet(PARQUET_DIR / "client_train.parquet")
client["client_id"] = client["client_id"].astype("string")
client["creation_date"] = pd.to_datetime(client["creation_date"], dayfirst=True, errors="raise")
client["target"] = client["target"].astype("int8")

client_overview = pd.DataFrame({
    "rows": [len(client)],
    "unique_clients": [client["client_id"].nunique()],
    "duplicate_rows": [client.duplicated().sum()],
    "missing_cells": [int(client.isna().sum().sum())],
    "fraud_rate": [client["target"].mean()],
})
display(client_overview)
display(client.head())
display(client.dtypes.rename("dtype").to_frame())

assert client["client_id"].is_unique, "Expected one row per client."
assert set(client["target"].unique()) <= {0, 1}, "Target must be binary."

development_ids, final_holdout_ids = train_test_split(
    client["client_id"],
    test_size=FINAL_HOLDOUT_SIZE,
    stratify=client["target"],
    random_state=RANDOM_STATE,
)
final_holdout_ids = set(final_holdout_ids)
client["split"] = np.where(client["client_id"].isin(final_holdout_ids), "final_holdout", "development")
display(pd.crosstab(client["split"], client["target"], margins=True))

In [ ]:
eda_client = client.query("split == 'development'").copy()
target_summary = (
    eda_client["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .to_frame("clients")
)
target_summary["share"] = target_summary["clients"] / len(eda_client)
display(target_summary)

ax = sns.countplot(data=eda_client, x="target", hue="target", legend=False)
ax.set(title="Strong class imbalance (development clients only)", xlabel="Fraud target", ylabel="Clients")
for container in ax.containers:
    ax.bar_label(container, fmt="%d")
plt.show()

## 2. Audit and aggregate invoice history

The source column names `disrict` and `counter_statue` are misspelled; they are preserved to match the supplied schema. The cell below computes exact additive/min/max client aggregates in chunks and records missing values and unexpected index direction without retaining all invoice rows.

Parquet is used here because it has a fixed column schema and can be read in record batches. That avoids the mixed-type CSV issue in `counter_statue` and keeps the browser kernel from attempting to hold the full invoice table at once.

In [ ]:
invoice_parquet = pq.ParquetFile(PARQUET_DIR / "invoice_train.parquet")
invoice_sample = invoice_parquet.read_row_group(0).to_pandas().head(5_000)
invoice_sample["invoice_date"] = pd.to_datetime(invoice_sample["invoice_date"], errors="raise")
display(invoice_sample.head())
display(invoice_sample.dtypes.rename("inferred_dtype").to_frame())
print(f"Sample shape: {invoice_sample.shape}")

In [ ]:
CONSUMPTION_COLUMNS = [f"consommation_level_{level}" for level in range(1, 5)]

def aggregate_invoices(path, chunk_size=250_000, as_of_date=None):
    partials = []
    row_hashes = []
    missing_counts = None
    status_counts = pd.Series(dtype="int64")
    tariff_counts = pd.Series(dtype="int64")
    source_rows = 0
    total_rows = 0
    excluded_after_cutoff = 0
    cutoff = pd.Timestamp(as_of_date) if as_of_date is not None else None
    started = time.perf_counter()

    parquet_file = pq.ParquetFile(path)
    reader = parquet_file.iter_batches(batch_size=chunk_size)

    for chunk_number, batch in enumerate(reader, start=1):
        chunk = batch.to_pandas()
        chunk["client_id"] = chunk["client_id"].astype("string")
        chunk["counter_statue"] = chunk["counter_statue"].astype("string")
        chunk["invoice_date"] = pd.to_datetime(chunk["invoice_date"], errors="raise")
        source_rows += len(chunk)
        if cutoff is not None:
            after_cutoff = chunk["invoice_date"].gt(cutoff)
            excluded_after_cutoff += int(after_cutoff.sum())
            chunk = chunk.loc[~after_cutoff].copy()
        total_rows += len(chunk)
        row_hashes.append(pd.util.hash_pandas_object(chunk, index=False).to_numpy())
        chunk_missing = chunk.isna().sum()
        missing_counts = chunk_missing if missing_counts is None else missing_counts.add(chunk_missing, fill_value=0)
        status_counts = status_counts.add(chunk["counter_statue"].value_counts(), fill_value=0)
        tariff_counts = tariff_counts.add(chunk["tarif_type"].value_counts(), fill_value=0)

        chunk["total_consumption"] = chunk[CONSUMPTION_COLUMNS].sum(axis=1)
        chunk["index_delta"] = chunk["new_index"] - chunk["old_index"]
        chunk["zero_consumption"] = chunk["total_consumption"].eq(0).astype("int8")
        chunk["is_elec"] = chunk["counter_type"].eq("ELEC").astype("int8")
        chunk["is_gaz"] = chunk["counter_type"].eq("GAZ").astype("int8")
        chunk["index_went_backwards"] = chunk["index_delta"].lt(0).astype("int8")

        grouped = chunk.groupby("client_id", observed=True).agg(
            invoice_count=("client_id", "size"),
            first_invoice=("invoice_date", "min"),
            last_invoice=("invoice_date", "max"),
            total_consumption_sum=("total_consumption", "sum"),
            total_consumption_max=("total_consumption", "max"),
            index_delta_sum=("index_delta", "sum"),
            index_delta_min=("index_delta", "min"),
            index_delta_max=("index_delta", "max"),
            months_sum=("months_number", "sum"),
            zero_consumption_count=("zero_consumption", "sum"),
            elec_invoice_count=("is_elec", "sum"),
            gaz_invoice_count=("is_gaz", "sum"),
            backwards_index_count=("index_went_backwards", "sum"),
        )
        partials.append(grouped.reset_index())
        if chunk_number % 5 == 0:
            print(f"Processed {total_rows:,} invoice rows...")

    partial = pd.concat(partials, ignore_index=True)
    client_agg = partial.groupby("client_id", as_index=False).agg(
        invoice_count=("invoice_count", "sum"),
        first_invoice=("first_invoice", "min"),
        last_invoice=("last_invoice", "max"),
        total_consumption_sum=("total_consumption_sum", "sum"),
        total_consumption_max=("total_consumption_max", "max"),
        index_delta_sum=("index_delta_sum", "sum"),
        index_delta_min=("index_delta_min", "min"),
        index_delta_max=("index_delta_max", "max"),
        months_sum=("months_sum", "sum"),
        zero_consumption_count=("zero_consumption_count", "sum"),
        elec_invoice_count=("elec_invoice_count", "sum"),
        gaz_invoice_count=("gaz_invoice_count", "sum"),
        backwards_index_count=("backwards_index_count", "sum"),
    )

    client_agg["mean_consumption"] = client_agg["total_consumption_sum"] / client_agg["invoice_count"]
    client_agg["mean_index_delta"] = client_agg["index_delta_sum"] / client_agg["invoice_count"]
    client_agg["mean_months"] = client_agg["months_sum"] / client_agg["invoice_count"]
    client_agg["zero_consumption_rate"] = client_agg["zero_consumption_count"] / client_agg["invoice_count"]
    client_agg["elec_share"] = client_agg["elec_invoice_count"] / client_agg["invoice_count"]
    client_agg["active_days"] = (client_agg["last_invoice"] - client_agg["first_invoice"]).dt.days

    invoice_hashes = pd.Series(np.concatenate(row_hashes), copy=False)
    audit = {
        "source_invoice_rows": source_rows,
        "invoice_rows": total_rows,
        "excluded_after_cutoff": excluded_after_cutoff,
        "duplicate_row_hash_matches": int(invoice_hashes.duplicated().sum()),
        "invoice_clients": client_agg["client_id"].nunique(),
        "missing_by_column": missing_counts.sort_values(ascending=False),
        "counter_status_counts": status_counts.sort_index().astype("int64"),
        "tariff_counts": tariff_counts.sort_index().astype("int64"),
        "elapsed_seconds": time.perf_counter() - started,
    }
    return client_agg, audit

In [ ]:
invoice_features, invoice_audit = aggregate_invoices(
    PARQUET_DIR / "invoice_train.parquet",
    chunk_size=CHUNK_SIZE,
    as_of_date=AS_OF_DATE,
)

print(f"Processed {invoice_audit['invoice_rows']:,} rows in {invoice_audit['elapsed_seconds']:.1f}s")
display(invoice_features.head())
display(invoice_audit["missing_by_column"].rename("missing_values").to_frame())
display(invoice_audit["counter_status_counts"].rename("invoice_rows").to_frame())
display(invoice_audit["tariff_counts"].rename("invoice_rows").to_frame())

In [ ]:
unknown_invoice_clients = set(invoice_features["client_id"]) - set(client["client_id"])
clients_without_invoices = set(client["client_id"]) - set(invoice_features["client_id"])
print(f"Invoice clients absent from client table: {len(unknown_invoice_clients):,}")
print(f"Clients without invoices: {len(clients_without_invoices):,}")

analysis = client.merge(
    invoice_features,
    on="client_id",
    how="left",
    validate="one_to_one",
)
analysis["customer_tenure_days_at_last_invoice"] = (
    analysis["last_invoice"] - analysis["creation_date"]
).dt.days

assert len(analysis) == len(client)
assert not unknown_invoice_clients
display(analysis.head())

eda_analysis = analysis.query("split == 'development'").copy()
print(f"EDA uses {len(eda_analysis):,} development clients; {analysis['split'].eq('final_holdout').sum():,} final-holdout clients remain excluded from target comparisons.")

## 3. Compare client history and behavior by target

These are associations, not causal explanations. Large differences may indicate signal, label-collection bias, or different observation windows. Review both medians and distributions because consumption variables are heavy-tailed.

The plots use `log1p` for count and consumption features so a handful of very large customers do not hide the overall pattern. Boxplots suppress extreme points visually, but the numeric summary still includes every development client. If a feature differs by target, ask whether it is available at the time an inspection decision would be made.

In [ ]:
eda_features = [
    "invoice_count",
    "active_days",
    "mean_consumption",
    "total_consumption_max",
    "zero_consumption_rate",
    "mean_index_delta",
    "mean_months",
    "elec_share",
    "customer_tenure_days_at_last_invoice",
]

summary_by_target = eda_analysis.groupby("target")[eda_features].agg(["median", "mean"])
display(summary_by_target.T)

plot_sample = eda_analysis.sample(n=min(40_000, len(eda_analysis)), random_state=RANDOM_STATE).copy()
plot_sample["log1p_invoice_count"] = np.log1p(plot_sample["invoice_count"])
plot_sample["log1p_mean_consumption"] = np.log1p(plot_sample["mean_consumption"].clip(lower=0))
plot_sample["log1p_max_consumption"] = np.log1p(plot_sample["total_consumption_max"].clip(lower=0))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, column, title in zip(
    axes,
    ["log1p_invoice_count", "log1p_mean_consumption", "zero_consumption_rate"],
    ["Invoice count (log1p)", "Mean consumption (log1p)", "Zero-consumption rate"],
):
    sns.boxplot(data=plot_sample, x="target", y=column, hue="target", legend=False, showfliers=False, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
def fraud_rate_by_group(frame, column, minimum_clients=50):
    result = frame.groupby(column, dropna=False)["target"].agg(clients="size", fraud_rate="mean")
    return result.query("clients >= @minimum_clients").sort_values("fraud_rate", ascending=False)

display(fraud_rate_by_group(eda_analysis, "client_catg"))
display(fraud_rate_by_group(eda_analysis, "region").head(20))

eda_analysis["history_length_bin"] = pd.qcut(
    eda_analysis["invoice_count"],
    q=5,
    duplicates="drop",
)
history_bias = fraud_rate_by_group(eda_analysis, "history_length_bin", minimum_clients=1)
display(history_bias)

ax = history_bias["fraud_rate"].plot(kind="bar", figsize=(9, 4), title="Fraud rate by invoice-history length")
ax.set(xlabel="Invoice-count quantile", ylabel="Observed fraud rate")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
quality_checks = pd.Series({
    "duplicate_client_rows": int(client.duplicated().sum()),
    "duplicate_client_ids": int(client["client_id"].duplicated().sum()),
    "missing_client_cells": int(client.isna().sum().sum()),
    "missing_invoice_cells": int(invoice_audit["missing_by_column"].sum()),
    "duplicate_invoice_row_hash_matches": invoice_audit["duplicate_row_hash_matches"],
    "clients_without_invoices": len(clients_without_invoices),
    "invoice_clients_without_client_row": len(unknown_invoice_clients),
    "clients_with_backwards_index": int(analysis["backwards_index_count"].gt(0).sum()),
    "creation_after_first_invoice": int(analysis["creation_date"].gt(analysis["first_invoice"]).sum()),
    "negative_customer_tenure": int(analysis["customer_tenure_days_at_last_invoice"].lt(0).sum()),
}, name="count")
display(quality_checks.to_frame())

display(analysis[["creation_date", "first_invoice", "last_invoice"]].agg(["min", "max"]))

## 4. Record findings before modeling

Complete this section together after running the notebook. Write short answers in a Markdown cell or issue; this becomes the evidence base for the baseline-model and slide decisions.

- **Data quality:** Which anomalies are real behavior, meter resets, corrections, or bad records?
- **Likely signal:** Which client/history/consumption features differ meaningfully by target?
- **Potential bias:** Does target rate depend strongly on region, client category, tenure, or amount of history?
- **Leakage risk:** Could any status/code be recorded only after a fraud investigation? Confirm feature availability at scoring time.
- **Stakeholder decision:** What is the weekly/monthly inspection capacity `k`, and what hit rate makes deployment valuable?

### Recommended next feature iteration

1. Add standard deviation and robust quantiles of consumption/index deltas.
2. Compare recent (for example, last 6–12 months) behavior with each client's earlier history.
3. Count meters, tariffs, statuses, and status changes per client.
4. Add time gaps, invoice regularity, trend, sudden-drop, and sudden-spike features.
5. Treat client/region/category codes as categorical; never use `client_id` as a predictor.
6. Build the Day 2 baseline before expanding further, then let error analysis determine the next features.

In [ ]:
if SAVE_FEATURES:
    output_dir = DATA_DIR / "processed"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "train_client_features.csv"
    analysis.to_csv(output_path, index=False)
    print(f"Saved {len(analysis):,} client rows to {output_path}")
else:
    print("Feature saving is disabled. Set SAVE_FEATURES = True when the table is ready to share locally.")